# Frozen TEST5000 — CONTROL
**T4 ×2**, Internet ON, Save&RunAll. Một notebook/một tài khoản. Secret `codex` của tài khoản chạy để tải checkpoint.bin đúng `ducanh2403/2-vi-cai-kia-qua-nhieu/180`, không tải output tài khoản kia. Nếu có mount exact checkpoint, điền WEIGHTS_ROOT, không cần API.
Gắn3Dataset: `csa-prompt-test-cache-v181` (share quyền cho Ngân), `vuthetam/mscoco-2014` (JPEG+JSON), `objectdetectionecache`. Không cần ROI134/175/visual-cache/UPT riêng, không train/cache lại.
CONTROL:normal raw+rerank; PROMPT:normal raw+rerank vàzero/shuffle(raw). Cùng checkpoint, giữROI/pairs/geometry/masks, donor ngoại ảnh khóahash42 theo từng loại slot. Beam5/max20, fixed .3decoder+.4CLIP+.3object, khôngBLIP/oracle/tuning hoặc fallback theo điểmTEST.
3h parent STOPcap, không phải ETA/quota; không tự chạy lại. Giữ output/5beams nếu scorer lỗi. Không optimizer/cập nhật trọng số. PriorTEST exposure và C10/P9 lịch XE ban đầu khác nhau được ghi trong kết quả. Zero/shuffle là độ nhạy khi suy luận, chưa phải đối chứng huấn luyện.
Tác giả chỉ kiểm static/NumPy/mock; model/CUDA/CLIP/scorer chưa được tác giả chạy. Gửi reviewZIP sau khi xong.


In [ ]:
import hashlib,json,os,queue,signal,subprocess,sys,tempfile,threading,time,traceback,urllib.request,zipfile,shutil
from pathlib import Path
ARM='CONTROL'
KERNEL_VERSION='ducanh2403/2-vi-cai-kia-qua-nhieu/180'

WEIGHTS_ROOT='' # Optional mounted exact checkpoint.bin; otherwise download own-account output.
SECRET_NAME='codex'
CACHE_ROOT=Path('/kaggle/input/datasets/ducanh2403/csa-prompt-test-cache-v181')
COCO_ROOT=Path('/kaggle/input/datasets/vuthetam/mscoco-2014')
COCO_JSON=COCO_ROOT/'dataset_coco.json'
DETECTIONS=Path('/kaggle/input/datasets/ducanh2403/objectdetectionecache/objectdetectioncache.json')
OUTPUT=Path('/kaggle/working/csa_prompt_TEST_'+ARM+'_v1')
WALL_HOURS=3 # STOPcap, not an ETA or billed GPU quota guarantee.
PARENT_DEADLINE=time.monotonic()+WALL_HOURS*3600
for p in (CACHE_ROOT,COCO_JSON,DETECTIONS):
    if not p.exists():raise FileNotFoundError('Attach3inputDatasets before GPU: '+str(p))
if OUTPUT.exists():raise RuntimeError('Preserve existing output; use a fresh runtime/output path')
if min(shutil.disk_usage('/tmp').free,shutil.disk_usage('/kaggle/working').free)<5*1024**3:raise RuntimeError('Need5GiB free for weights/models/output')
OUTPUT.mkdir(parents=True);SOURCE=Path(tempfile.mkdtemp(prefix='csa-prompt-test-'+ARM.lower()+'-',dir='/tmp'));LOG=OUTPUT/'notebook.log'
REVISION='ee72401571ed1959d67af5a6f2429ba774cdfc54'
MAPPING={'experiments/csa_paper/RERANK_TEST_INPUTS.json': 'RERANK_TEST_INPUTS.json', 'experiments/csa_paper/PROMPT_SCST_RESULT.json': 'PROMPT_SCST_RESULT.json', 'experiments/csa_paper/PROMPT_TEST_PREPARATION_RESULT.json': 'PROMPT_TEST_PREPARATION_RESULT.json', 'experiments/csa_paper/PROMPT_TEST_PROTOCOL.md': 'PROMPT_TEST_PROTOCOL.md', 'experiments/csa_paper/cuda_runtime_guard.py': 'cuda_runtime_guard.py', 'experiments/csa_paper/make_prompt_test_notebooks.py': 'make_prompt_test_notebooks.py', 'experiments/csa_paper/make_smoke_notebook.py': 'make_smoke_notebook.py', 'experiments/csa_paper/make_prompt_scst_notebooks.py': 'make_prompt_scst_notebooks.py', 'experiments/csa_paper/test_prompt_test_data.py': 'test_prompt_test_data.py', 'experiments/csa_paper/prompt_test_data.py': 'prompt_test_data.py', 'experiments/csa_paper/prompt_adapter_data.py': 'prompt_adapter_data.py', 'captioning/upt_prompts.py': 'captioning/upt_prompts.py', 'captioning/upt_action_text.py': 'captioning/upt_action_text.py', 'captioning/selected_relation_prompts.py': 'captioning/selected_relation_prompts.py', 'captioning/supervised_relations.py': 'captioning/supervised_relations.py', 'experiments/csa_paper/real_bank_data.py': 'real_bank_data.py', 'experiments/csa_paper/run_prompt_test.py': 'run_prompt_test.py', 'experiments/csa_paper/saved_quality_data.py': 'saved_quality_data.py', 'experiments/csa_paper/prompt_scst_data.py': 'prompt_scst_data.py', 'experiments/csa_paper/prompt_scratch_data.py': 'prompt_scratch_data.py', 'experiments/csa_paper/full_prompt_cache_data.py': 'full_prompt_cache_data.py', 'experiments/csa_paper/audit_assets.py': 'audit_assets.py', 'experiments/csa_paper/data.py': 'data.py', 'experiments/csa_paper/sharing_data.py': 'sharing_data.py', 'experiments/csa_paper/full_cache_data.py': 'full_cache_data.py', 'experiments/csa_paper/package_full_roi_dataset.py': 'package_full_roi_dataset.py', 'experiments/csa_paper/scst_pilot_data.py': 'scst_pilot_data.py', 'experiments/csa_paper/scst_full_data.py': 'scst_full_data.py', 'experiments/csa_paper/run_scst_pilot.py': 'run_scst_pilot.py', 'captioning/object_reward.py': 'captioning/object_reward.py', 'captioning/object_consistency.py': 'captioning/object_consistency.py', 'captioning/scst.py': 'captioning/scst.py', 'experiments/csa_paper/scst_pilot_model.py': 'scst_pilot_model.py', 'experiments/csa_paper/model.py': 'model.py', 'experiments/csa_paper/full_xe_data.py': 'full_xe_data.py', 'experiments/csa_paper/full_xe_decode.py': 'full_xe_decode.py', 'experiments/csa_paper/prompt_adapter_model.py': 'prompt_adapter_model.py', 'experiments/csa_paper/prompt_scst_rerank.py': 'prompt_scst_rerank.py', 'captioning/multiscore_reranking.py': 'captioning/multiscore_reranking.py', 'captioning/__init__.py': 'captioning/__init__.py'}
HASHES={'experiments/csa_paper/RERANK_TEST_INPUTS.json': '33395ec35a0803ebf711737f0bdba3b6be8c16244f4c292057658e0fa3feaf46', 'experiments/csa_paper/PROMPT_SCST_RESULT.json': '88d6ee2649dcc6f95a916539b5a07e88db4df02d7ec810e08942c31ab248523a', 'experiments/csa_paper/PROMPT_TEST_PREPARATION_RESULT.json': 'd99891923634188aa2d75185f0ef9894e3ab9e3be780a5a17e411076c3c9d205', 'experiments/csa_paper/PROMPT_TEST_PROTOCOL.md': '002277c7dab3880148d1d0e8c600ca117be78734f85b55c6dd8bf13d50485c70', 'experiments/csa_paper/cuda_runtime_guard.py': '640b5debc81607e3504930e4042571636cc3fbd525a6d0e5eb45abdd141fb773', 'experiments/csa_paper/make_prompt_test_notebooks.py': '0b274c9a9b9b208d496ea8dec539fffb17d610e55fad5be557e397e9de0ae175', 'experiments/csa_paper/make_smoke_notebook.py': '9de13e286efb3ddd29a2eed331219a9e60dcb384fc2f28e062f64dd8dd1c6054', 'experiments/csa_paper/make_prompt_scst_notebooks.py': 'cc5f491c5da80c1615736198a65d6f5b5259455867ee4dc4f6a8e55f6d5cde6c', 'experiments/csa_paper/test_prompt_test_data.py': 'c0565b2420d017fa5f251060d88f82d5a6f2e32fd2de98ee19b6b4bdcc566a5b', 'experiments/csa_paper/prompt_test_data.py': 'b2d5b530c46d733b00deb965ae1c9228dbb2232b93aa815b11d412f92ba1677a', 'experiments/csa_paper/prompt_adapter_data.py': 'd8b8b61a518d9d4e2d413386d08e60963586bf3b17a16e5c96ee3f54eeaf36c0', 'captioning/upt_prompts.py': '25bb24346d2b2c649ee7091c72d5def060ddf2c290524a35ef3d301d274fb65c', 'captioning/upt_action_text.py': 'ca0d8f1dc75be3ccfd589fcddc962e87a43222751a4d1beb68098cfa213d526c', 'captioning/selected_relation_prompts.py': 'de13dfbd80e4d4b8c08cd3e40407d5134cb1a5108c58f3eebfd4b74e237179d2', 'captioning/supervised_relations.py': '3e29fd60bbf166ae078581fb9ad34da6ce4462ee287f1c4eb9bbda2abd91f984', 'experiments/csa_paper/real_bank_data.py': '2c791f8dd856198ec1954a2edd1f9e46520d422f20eff07220e462fc2c9bd9a0', 'experiments/csa_paper/run_prompt_test.py': 'cf071bdf0a641ac3e5a9204b4442b8fc30c99ab1939dbbdca25c7eb75514b935', 'experiments/csa_paper/saved_quality_data.py': 'f8a092506e3adf96292203190ea2f83de397d3028a3ddc6293d58fd360199a1e', 'experiments/csa_paper/prompt_scst_data.py': '68e6e77a6b2f0d095ce86eb0fcf36b8829aa877ef6e2cfd2623a6fc2d1b1708f', 'experiments/csa_paper/prompt_scratch_data.py': '025496a549e12a49104992e1c48f09968ebf4dcb520d649c818df6b39c0a6959', 'experiments/csa_paper/full_prompt_cache_data.py': 'efb21689e1983e8ab22f9d63068cf262f58344498d412056580c8af54c481f16', 'experiments/csa_paper/audit_assets.py': '5be2fdf1ee094ef8b23a547b16f423ed3603f7c80b2fe5f2d46cb9224cb90959', 'experiments/csa_paper/data.py': '5a5bb451dc20184d3ea597d4dc8415a120583571f4e1a4dac75e29a306213a71', 'experiments/csa_paper/sharing_data.py': '68a065f5c055a55b41eae0ea554fbb7f8fd1124e8f5422679fcb1d4878f14829', 'experiments/csa_paper/full_cache_data.py': '58cf4d08310fe03e01441af2ddeb12fd3404f59e82259194a32fe98c8b002bc1', 'experiments/csa_paper/package_full_roi_dataset.py': 'be3e7d0f96add450a5295add6466e19977da90f486e5e726e3ea40a505cceecb', 'experiments/csa_paper/scst_pilot_data.py': '119324b3933e07a07ff5aa60a4cb4272ebfdd587d0451e56f338296954a2743b', 'experiments/csa_paper/scst_full_data.py': 'f1d88bd3b5bdf5f7e873b128a9624ee645db7b9c075824782572c7e87a87240b', 'experiments/csa_paper/run_scst_pilot.py': '9a640766bd063d2e5d5854dab9177168ae6528bf0fcd43467ba7e5b38618285e', 'captioning/object_reward.py': '23282b1faca0b8e177ad8ce300f0b78ad4a5ab7b100e4b359b044025c567800c', 'captioning/object_consistency.py': '5fbd75b0ff4a35f0475d460a294894009d6007f468e248a0e4b528aaa8fabf73', 'captioning/scst.py': '3a9a1f32fb27dae0447066e1388a31461f3f0781d52db5861880b037a572a78e', 'experiments/csa_paper/scst_pilot_model.py': '0b5e2d384ffc7195c82888f3f7904eac145ba7291ebed27dc8342ed8e1ac9576', 'experiments/csa_paper/model.py': '139edbb1b73caf3dadaf5c960b25542e282f88baf2ce9bdf141de922ceacbc09', 'experiments/csa_paper/full_xe_data.py': 'a7a5a3b4d70c9d3a357aa464dfe397a6cba8d851e33065299fb98721ddca7135', 'experiments/csa_paper/full_xe_decode.py': 'b8e618559feec6361cfb3b4619899e4968ab71ab67e113ff38b6d3cd99cdc235', 'experiments/csa_paper/prompt_adapter_model.py': '83e5fdb2662940ded06e56df3a0fdf512dc1046b2352e563b59f22a6117e245c', 'experiments/csa_paper/prompt_scst_rerank.py': '2f998867793a3abc3efcaac7836ad178e1521b4a48b71b7171849b2bf9791dc7', 'captioning/multiscore_reranking.py': 'd1baa60e244c86b32e721486d7923ead1391c3a2fb744651c257995f2d0bb443', 'captioning/__init__.py': 'ad259f50a32265d298d027a888f801d378c4669313b8dab06e0653978ce30633'}

for original,destination in MAPPING.items():
    if time.monotonic()>=PARENT_DEADLINE:raise TimeoutError('Source download deadline')
    raw=urllib.request.urlopen('https://raw.githubusercontent.com/Supzxjee/Image_Captioning/'+REVISION+'/'+original,timeout=90).read()
    if hashlib.sha256(raw).hexdigest()!=HASHES[original]:raise ValueError('Changed pinned source:'+original)
    p=SOURCE/destination;p.parent.mkdir(parents=True,exist_ok=True);p.write_bytes(raw)
(OUTPUT/'source_receipt.json').write_text(json.dumps(dict(commit=REVISION,sha256=HASHES),indent=2),encoding='utf-8')
os.environ['OMP_NUM_THREADS']='1';os.environ['PYTHONPATH']=str(SOURCE)
CHAIR=SOURCE/'chair';DEPS=SOURCE/'deps';CONFIG=SOURCE/'config.json'
cfg=dict(arm=ARM,cache_root=str(CACHE_ROOT),checkpoint_root=WEIGHTS_ROOT,dataset_json=str(COCO_JSON),coco_root=str(COCO_ROOT),detections=str(DETECTIONS),
    output=str(OUTPUT),chair=str(CHAIR),model_cache=str(SOURCE/'hf-cache'),decode_seconds=5400,rerank_seconds=1800)
def write_config():
    CONFIG.write_text(json.dumps(cfg,indent=2),encoding='utf-8');(OUTPUT/'protocol.json').write_text(json.dumps(cfg,indent=2),encoding='utf-8')
write_config()


In [ ]:
def run(args, timeout=1800):
    if time.monotonic()>=PARENT_DEADLINE:raise TimeoutError('Parent deadline exceeded')
    timeout=min(timeout,max(.01,PARENT_DEADLINE-time.monotonic()))
    print('Running:', ' '.join(map(str, args)), flush=True)
    process = subprocess.Popen(list(map(str, args)), cwd=SOURCE, stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT, text=True, errors='replace', bufsize=1, start_new_session=True)
    messages = queue.Queue()
    def reader():
        try:
            for line in process.stdout: messages.put(line)
        finally: messages.put(None)
    thread = threading.Thread(target=reader, daemon=True); thread.start()
    started = heartbeat = time.monotonic(); ended = False
    with LOG.open('a', encoding='utf-8') as log:
        try:
            while not ended or process.poll() is None or not messages.empty():
                try:
                    line = messages.get(timeout=1)
                    if line is None: ended = True
                    else: print(line, end='', flush=True); log.write(line); log.flush()
                except queue.Empty: pass
                now = time.monotonic()
                if now-heartbeat >= 30:
                    print('Prompt SCST active;', round(now-started), 'seconds', flush=True); heartbeat=now
                if now-started > timeout: raise TimeoutError('Parent/phase deadline exceeded')
            if process.wait(): raise subprocess.CalledProcessError(process.returncode, args)
        finally:
            if process.poll() is None:
                os.killpg(process.pid, signal.SIGTERM)
                try: process.wait(timeout=10)
                except subprocess.TimeoutExpired: os.killpg(process.pid, signal.SIGKILL); process.wait()
            thread.join(timeout=5)


In [ ]:
complete=False
try:
    run([sys.executable,'-u',SOURCE/'test_prompt_test_data.py'],timeout=120)
    run([sys.executable,'-u',SOURCE/'cuda_runtime_guard.py','--output',OUTPUT],timeout=60)
    if not WEIGHTS_ROOT:
        run([sys.executable,'-m','pip','install','-q','kaggle==2.2.5'],timeout=300)
        # Local mock checks exact version routing before the authenticated download.
        route="from types import SimpleNamespace;from unittest.mock import patch;from kaggle.api.kaggle_api_extended import KaggleApi;import tempfile;api=KaggleApi();owner,slug,version="+repr(KERNEL_VERSION)+".split('/');ctx=tempfile.TemporaryDirectory();d=ctx.__enter__();factory=patch.object(api,'build_kaggle_client');f=factory.__enter__();ep=f.return_value.__enter__.return_value.kernels.kernels_api_client.list_kernel_session_output;ep.return_value=SimpleNamespace(files=[],next_page_token=None,log=None);api.kernels_output("+repr(KERNEL_VERSION)+",path=d);r=ep.call_args.args[0];assert (r.user_name,r.kernel_slug,r.version_label)==(owner,slug,'v'+version);factory.__exit__(None,None,None);ctx.__exit__(None,None,None);print('Exact version route mock PASS')"
        from kaggle_secrets import UserSecretsClient
        token=UserSecretsClient().get_secret(SECRET_NAME).strip()
        if not token:raise RuntimeError('Enable configured Kaggle API Secret for THIS account')
        previous={k:os.environ.pop(k,None) for k in ('KAGGLE_API_TOKEN','KAGGLE_USERNAME','KAGGLE_KEY')}
        try:
            os.environ['KAGGLE_API_TOKEN']=token
            run([sys.executable,'-c',route],timeout=60)
            weight_dir=OUTPUT/'source_xe';weight_dir.mkdir()
            run([sys.executable,'-c','from kaggle.cli import main;main()','kernels','output',KERNEL_VERSION,
                 '-p',weight_dir,'--file-pattern',r'(^|/)checkpoint\.bin$'],timeout=900)
        finally:
            for k in previous:os.environ.pop(k,None)
            for k,v in previous.items():
                if v is not None:os.environ[k]=v
            del token
        cfg['checkpoint_root']=str(weight_dir)
    sys.path.insert(0,str(SOURCE))
    from prompt_test_data import source_checkpoint
    print('Exact selected checkpoint:',source_checkpoint(cfg['checkpoint_root'],ARM),flush=True);write_config()
    run([sys.executable,'-m','pip','install','-q','pycocoevalcap==1.2','nltk==3.9.2','future'],timeout=300)
    run([sys.executable,'-m','pip','install','-q','--no-deps','git+https://github.com/clips/pattern.git@af754685cca3713db0abc4f020f2e94467c19d85'],timeout=300)
    run([sys.executable,'-c',"import nltk;from pattern.en import singularize;assert singularize('umbrellas')=='umbrella';assert nltk.download('punkt');assert nltk.download('punkt_tab')"],timeout=300)
    run(['java','-version'],timeout=30)
    run(['git','init',CHAIR],timeout=60)
    run(['git','-C',CHAIR,'remote','add','origin','https://github.com/Maxlinn/CHAIR-metric-standalone.git'],timeout=60)
    run(['git','-C',CHAIR,'fetch','--depth','1','origin','4087a26211aa2339b9a76307cb8f0321ef691d0a'],timeout=300)
    run(['git','-C',CHAIR,'checkout','--detach','FETCH_HEAD'],timeout=60)
    # Fail inference dependencies/assets before expensive training; isolate HF packages from host apps.
    run([sys.executable,'-m','pip','install','-q','--no-deps','--target',DEPS,'transformers==4.57.1','tokenizers==0.22.1',
         'huggingface-hub==0.36.2','safetensors==0.6.2'],timeout=300)
    os.environ['PYTHONPATH']=str(DEPS)+os.pathsep+str(SOURCE)
    run([sys.executable,'-c',"from importlib.metadata import version;from transformers import CLIPModel,CLIPProcessor;assert version('transformers')=='4.57.1';assert version('huggingface-hub')=='0.36.2';print('Isolated HF imports PASS')"],timeout=60)
    run([sys.executable,'-u',SOURCE/'prompt_scst_rerank.py','prefetch','--config',CONFIG],timeout=1200)

    run([sys.executable,'-u',SOURCE/'run_prompt_test.py','prepare','--config',CONFIG],timeout=1800)
    run([sys.executable,'-u',SOURCE/'run_prompt_test.py','metric_preflight','--config',CONFIG],timeout=240)
    launch=[sys.executable,'-u','-m','torch.distributed.run','--standalone','--nnodes=1','--nproc_per_node=2','--max_restarts=0']
    cfg['decode_seconds']=PARENT_DEADLINE-time.monotonic()-2400
    if cfg['decode_seconds']<1200:raise TimeoutError('Insufficient decode/score reserve after preparation')
    write_config();run([*launch,SOURCE/'run_prompt_test.py','decode','--config',CONFIG],timeout=cfg['decode_seconds']+30)
    cfg['rerank_seconds']=PARENT_DEADLINE-time.monotonic()-1200
    if cfg['rerank_seconds']<180:raise TimeoutError('Captions saved; insufficient rerank reserve, do not regenerate')
    write_config();run([*launch,SOURCE/'run_prompt_test.py','rerank','--config',CONFIG],timeout=cfg['rerank_seconds']+30)
    run([sys.executable,'-u',SOURCE/'run_prompt_test.py','score','--config',CONFIG],timeout=1200)
    complete=json.loads((OUTPUT/'result.json').read_text(encoding='utf-8'))['status']=='PROMPT_TEST5000_COMPLETE'
except BaseException:
    (OUTPUT/'failure.txt').write_text(traceback.format_exc(),encoding='utf-8');raise
finally:
    (OUTPUT/'run_status.json').write_text(json.dumps(dict(complete=complete,arm=ARM,training=False,test_tuning=False)),encoding='utf-8')
    REVIEW=OUTPUT.with_name(OUTPUT.name+'_review.zip')
    with zipfile.ZipFile(REVIEW,'w',zipfile.ZIP_DEFLATED) as z:
        for p in OUTPUT.iterdir():
            if p.is_file() and p.suffix in ('.json','.jsonl','.log','.txt'):z.write(p,p.name)
    print('SEND REVIEW:',REVIEW,'\nKEEP WHOLE OUTPUT:',OUTPUT,flush=True)
    print('TEST COMPLETE; all locked conditions reported.' if complete else 'INCOMPLETE; preserve captions/logs, no automatic retry or policy change.',flush=True)
